# ⏱️ RS-LiDAR & Vanilla LiDAR: Benchmark Table 2 Efficiency (Time & Peak VRAM on A100)

> **Mục tiêu**: Đo đạc độc lập, chuẩn xác thời gian thực thi phân rã 3 thành phần (khớp Bảng 9) và đỉnh VRAM (Bảng 2) của **RS-LiDAR** và **Vanilla LiDAR** trên GPU **NVIDIA A100**:
> 1. **$T_{\text{lookahead}}$**: Thời gian sinh $N$ hạt lookahead (DPM-5 hoặc DMD-1) và giải mã VAE.
> 2. **$T_{\text{reward}}$**: Thời gian đánh giá ImageReward (**Full-Batch 1 forward duy nhất** trên trọn vẹn $M \times N$ ảnh).
> 3. **$T_{\text{target}}$**: Thời gian khử nhiễu mục tiêu (Closed-form steering, sinh 4 ảnh đích, **DỪNG NGAY KHÔNG CHẤM REWARD Ở PHASE 2**).
> 4. **$T_{\text{total}}$**: $T_{\text{lookahead}} + T_{\text{reward}} + T_{\text{target}}$.
> 5. **Peak VRAM**: $\text{torch.cuda.max\_memory\_allocated()} / 1024^3$ (GiB).
> 6. **Auto Google Drive Sync**: Tự động lưu và đồng bộ tức thì file kết quả CSV & JSON về Google Drive cá nhân sau từng prompt.

---

### 📌 3 Cấu Hình Benchmark Chuẩn Bảng 2:
| Setting Key | Model | Lookahead Solver | Target Solver | Scale | $\eta$ |
| :--- | :--- | :--- | :--- | :---: | :---: |
| `sdv1.5_ddim50` | SD v1.5 | DPM-5 ($N=50$) | DDIM 50 steps ($N=4$) | 12.5 | 0.0 |
| `sdv1.5_ddpm100` | SD v1.5 | DPM-5 ($N=50$) | DDPM 100 steps ($N=4$) | 12.5 | 1.0 |
| `sdxl_dmd1` | SDXL (2.6B) | DMD-1 ($N=100$, 1 bước) | DDPM 100 steps ($N=4$) | 8.0 | 1.0 |

## 1. 🖥️ Kiểm Tra Phần Cứng GPU (Khuyến nghị NVIDIA A100)

In [ ]:
!nvidia-smi

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n✅ GPU khả dụng: {gpu_name} ({total_vram:.2f} GiB VRAM)")
    if "A100" in gpu_name:
        print("🌟 Bạn đang chạy trên NVIDIA A100 chuẩn benchmark bài báo ICML 2026!")
    else:
        print("ℹ️ Lưu ý: GPU hiện tại không phải A100. Thời gian thực tế có thể chênh lệch so với Bảng 2 bài báo.")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU (A100).")

## 2. 📁 Gắn Kết Google Drive Cá Nhân (Tự Động Lưu Kết Quả Chống Mất Dữ Liệu)

In [ ]:
# @title 📁 Gắn kết Google Drive & Thiết lập thư mục lưu trữ
import os
from google.colab import drive

MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_BENCHMARK_DIR = None

if MOUNT_DRIVE:
    try:
        drive.mount('/content/drive')
        base_drive = None
        for cand in ['/content/drive/MyDrive', '/content/drive/My Drive', '/content/drive']:
            if os.path.exists(cand):
                base_drive = cand
                break
        if base_drive is None:
            base_drive = '/content/drive/MyDrive'

        DRIVE_BENCHMARK_DIR = os.path.join(base_drive, "RS-LiDAR", "results", "benchmark_efficiency")
        os.makedirs(DRIVE_BENCHMARK_DIR, exist_ok=True)
        print(f"\n✅ Đã gắn kết Google Drive thành công!")
        print(f"📁 Thư mục tự động sao lưu trên Drive: {DRIVE_BENCHMARK_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể mount Google Drive: {e}. Kết quả sẽ chỉ được lưu tạm thời trên đĩa Colab.")
        DRIVE_BENCHMARK_DIR = None
else:
    print("ℹ️ Bỏ qua gắn kết Google Drive. Kết quả sẽ lưu cục bộ tại 'results/benchmark_efficiency/'.")

## 3. 📂 Đồng Bộ Mã Nguồn Từ GitHub & Định Tuyến Thư Mục

In [ ]:
# @title 📥 Bước 3: Clone / Update Repository RS-LiDAR và Chuyển Thư Mục
import os

!git clone https://github.com/leekwanreal/RS-LiDAR.git /content/RS-LiDAR || git -C /content/RS-LiDAR pull origin main
%cd /content/RS-LiDAR

assert os.path.exists("benchmark_table2_efficiency.py"), f"❌ Chưa tìm thấy benchmark_table2_efficiency.py trong {os.getcwd()}!"
print(f"\n✅ Đã ở đúng thư mục làm việc: {os.getcwd()}")


## 4. 📦 Cài Đặt Thư Viện Cần Thiết

In [ ]:
# 1. Tat backend TensorFlow de loai bo xung dot Protobuf runtime_version
import os, urllib.request
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# 2. Nang cap protobuf va cai dat cac thu vien tuong thich PyTorch 2.x chuan Table 2 Replication
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Dam bao file tu dien OpenCLIP cho HPSv2
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), 'src', 'open_clip', 'bpe_simple_vocab_16e6.txt.gz')
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        print('📥 Dang tai file BPE vocab cho HPSv2...')
        urllib.request.urlretrieve('https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz', hpsv2_vocab)
except Exception:
    pass

print('\n✅ Cai dat moi truong chuan xac 100% tu Table 2 Replication (Khong xung dot Protobuf)!')


## 5. ⚙️ Cấu Hình Thực Nghiệm (Interactive Form)

In [ ]:
# @title 🎛️ Chọn Tham Số Benchmark

# @markdown **1. Cấu hình bài báo (Setting)**:
SETTING = "sdv1.5_ddpm100" # @param ["sdv1.5_ddim50", "sdv1.5_ddpm100", "sdxl_dmd1"]

# @markdown **2. Phương pháp lấy mẫu (Method)**:
METHOD = "rs-lidar" # @param ["rs-lidar", "lidar"]

# @markdown **3. Số lượng prompts đo trung bình**:
NUM_PROMPTS = 3 # @param {type:"integer"}

# @markdown **4. Độ lệch chuẩn nhiễu làm mịn (Sigma)**:
SIGMA = 1.0 # @param {type:"number"}
# @markdown *(Gợi ý: 1.0 cho SD 1.5; 0.5 cho SDXL; Tự động về 0.0 nếu chọn Vanilla LiDAR)*

# @markdown **5. Số mẫu Monte Carlo (M)**:
NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# @markdown *(Gợi ý: 4 cho RS-LiDAR; Tự động về 1 nếu chọn Vanilla LiDAR)*

# @markdown **6. Batch Size ImageReward (Mặc định: Full Batch)**:
REWARD_BATCH_SIZE = "None" # @param ["None", "64", "32", "16"]
# @markdown *(Giữ 'None' để ép chạy trọn vẹn 1 batch duy nhất trên A100!)*

# @markdown **7. Bật Warm-up GPU (1 prompt khởi động kernel)**:
ENABLE_WARMUP = True # @param {type:"boolean"}

# @markdown **8. Tự động sao lưu kết quả sang Google Drive**:
SAVE_TO_DRIVE = True # @param {type:"boolean"}

print(f"Cấu hình đã chọn:")
print(f"• Setting:           {SETTING}")
print(f"• Method:            {METHOD.upper()}")
print(f"• Prompts to test:   {NUM_PROMPTS}")
print(f"• Sigma:             {SIGMA if METHOD == 'rs-lidar' else 0.0}")
print(f"• Monte Carlo M:     {NUM_MC_SAMPLES if METHOD == 'rs-lidar' else 1}")
print(f"• Reward Batch Size: {REWARD_BATCH_SIZE}")
print(f"• GPU Warmup:        {ENABLE_WARMUP}")
print(f"• Save to Drive:     {SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR is not None}")

## 6. 🚀 Thực Thi Benchmark Table 2 Efficiency

In [ ]:
import os

# Đảm bảo đứng đúng thư mục chứa benchmark script
if not os.path.exists("benchmark_table2_efficiency.py"):
    %cd /content/RS-LiDAR

warmup_flag = "--warmup" if ENABLE_WARMUP else "--no_warmup"
cmd = [
    "python", "benchmark_table2_efficiency.py",
    "--setting", SETTING,
    "--method", METHOD,
    "--num_prompts", str(NUM_PROMPTS),
    "--sigma", str(SIGMA),
    "--num_mc_samples", str(NUM_MC_SAMPLES),
    "--reward_batch_size", str(REWARD_BATCH_SIZE),
    warmup_flag,
    "--output_dir", "results/benchmark_efficiency"
]

# Thêm cờ đồng bộ Google Drive nếu có
if SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR is not None:
    cmd.extend(["--drive_backup_dir", f"'{DRIVE_BENCHMARK_DIR}'"])

print(f"Chạy lệnh: {' '.join(cmd)}\n")
!{' '.join(cmd)}

## 7. 📊 Hiển Thị Bảng Phân Rã Thời Gian (Khớp Table 9 & Table 2 Bài Báo Gốc)

In [ ]:
import os, glob, json
import pandas as pd
from tabulate import tabulate
from IPython.display import display, HTML

# Baseline tham chiếu chính thức từ bài báo gốc ICML 2026 (Table 9 & Table 2)
PAPER_BASELINES = {
    "sdv1.5_ddim50": {
        "name": "SD 1.5 (DDIM-50 / n=50)",
        "t_lookahead": 1.10,
        "t_reward": 0.65,
        "t_target": 3.51,
        "t_total": 5.26,
        "peak_vram": 8.90,
    },
    "sdv1.5_ddpm100": {
        "name": "SD 1.5 (DDPM-100 / n=50)",
        "t_lookahead": 1.10,
        "t_reward": 0.65,
        "t_target": 7.02,
        "t_total": 8.77,
        "peak_vram": 8.90,
    },
    "sdxl_dmd1": {
        "name": "SDXL (DMD-1 / n=100)",
        "t_lookahead": 4.30,
        "t_reward": 1.30,
        "t_target": 50.00,
        "t_total": 55.60,
        "peak_vram": 33.84,
    },
}

# Ưu tiên đọc từ Google Drive nếu có sao lưu, fallback về local
search_dirs = []
if DRIVE_BENCHMARK_DIR and os.path.exists(DRIVE_BENCHMARK_DIR):
    search_dirs.append(DRIVE_BENCHMARK_DIR)
search_dirs.append("results/benchmark_efficiency")

json_files = []
for s_dir in search_dirs:
    json_files.extend(glob.glob(os.path.join(s_dir, f"efficiency_{SETTING}_{METHOD}_*.json")))
if not json_files:
    for s_dir in search_dirs:
        json_files.extend(glob.glob(os.path.join(s_dir, "*.json")))

if json_files:
    latest_json = max(json_files, key=os.path.getmtime)
    with open(latest_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    ref = PAPER_BASELINES.get(SETTING, {})

    rows = [
        [
            f"Vanilla LiDAR (Paper)",
            f"{ref.get('t_lookahead', '-'):.2f}s",
            f"{ref.get('t_reward', '-'):.2f}s",
            f"{ref.get('t_target', '-'):.2f}s",
            f"{ref.get('t_total', '-'):.2f}s",
            f"{ref.get('peak_vram', '-'):.2f} GiB",
        ],
        [
            f"<b>{METHOD.upper()} (Measured)</b>",
            f"<b>{data['mean_t_lookahead_sec']:.2f}s</b>",
            f"<b>{data['mean_t_reward_sec']:.2f}s</b>",
            f"<b>{data['mean_t_target_sec']:.2f}s</b>",
            f"<b>{data['mean_t_total_sec']:.2f}s</b>",
            f"<b>{data['max_peak_vram_gib']:.2f} GiB</b>",
        ]
    ]

    headers = ["Method", "T_lookahead", "T_reward", "T_target", "T_total / prompt", "Peak VRAM"]
    df_summary = pd.DataFrame(rows, columns=headers)

    print(f"\n{'='*80}")
    print(f"📊 KẾT QUẢ ĐỐI CHIẾU TABLE 9 & TABLE 2 (Setting: {SETTING})")
    print(f"{'='*80}")
    print(tabulate(rows, headers=headers, tablefmt="grid"))
    print(f"{'='*80}\n")

    # Hiển thị bảng HTML rich
    display(HTML(f"<h3>📊 Bảng Đối Chiếu Thời Gian & Bộ Nhớ: {ref.get('name', SETTING)}</h3>"))
    display(HTML(df_summary.to_html(escape=False, index=False)))
    if DRIVE_BENCHMARK_DIR:
        print(f"☁️ Kết quả đã được lưu vĩnh viễn trên Google Drive: {DRIVE_BENCHMARK_DIR}")
else:
    print("⚠️ Chưa tìm thấy file kết quả trong 'results/benchmark_efficiency/'.")

## 8. 🔄 Chạy Tự Động So Sánh Trực Tiếp Cả 2 (Vanilla LiDAR vs RS-LiDAR)
> Cell này sẽ tự động chạy liên tiếp cả **Vanilla LiDAR** và **RS-LiDAR** trên cùng setting và đồng bộ cả 2 về Google Drive.

In [ ]:
# @title ⚡ Auto Comparative Run (LiDAR vs RS-LiDAR)
RUN_COMPARISON = False # @param {type:"boolean"}

if RUN_COMPARISON:
    if not os.path.exists("benchmark_table2_efficiency.py"):
        %cd /content/RS-LiDAR

    print("\n🚀 BẮT ĐẦU CHẠY SO SÁNH 2 PHƯƠNG PHÁP...")
    drive_opt = f"--drive_backup_dir '{DRIVE_BENCHMARK_DIR}'" if (SAVE_TO_DRIVE and DRIVE_BENCHMARK_DIR) else ""
    
    # 1. Chạy Vanilla LiDAR
    print("\n--- [1/2] Đang chạy Vanilla LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method lidar --num_prompts {NUM_PROMPTS} --reward_batch_size None --warmup {drive_opt}
    
    # 2. Chạy RS-LiDAR
    print("\n--- [2/2] Đang chạy RS-LiDAR ---")
    !python benchmark_table2_efficiency.py --setting {SETTING} --method rs-lidar --num_prompts {NUM_PROMPTS} --sigma {SIGMA} --num_mc_samples {NUM_MC_SAMPLES} --reward_batch_size None --warmup {drive_opt}
    
    print("\n✅ Hoàn thành so sánh cả 2 phương pháp! Chạy lại Cell 7 để xem bảng tổng hợp đối chiếu.")
else:
    print("Đánh dấu tích vào RUN_COMPARISON để kích hoạt chạy so sánh tự động.")